In [1]:
# test1 FeedbackDetector
import numpy as np
from feedback_detector import FeedbackDetector

# -----------------------------
# Dummy InstrumentNode class
# -----------------------------
class InstrumentNode:
    def __init__(self, name, instrument):
        self.name = name
        self.instrument = instrument
        self.fft_history = []

    def update_frame(self, fft_data):
        self.fft_history.append(fft_data)


# -----------------------------
# Setup
# -----------------------------
sample_rate = 44100
N = 4096
freqs = np.fft.rfftfreq(N, d=1.0/sample_rate)

detector = FeedbackDetector(threshold_db_per_frame=1.5, window_frames=3)


# -----------------------------
# Helper functions
# -----------------------------
def generate_stable_fft():
    return np.random.uniform(0.0001, 0.001, len(freqs))


def generate_feedback_fft(base, spike_idx, growth_factor):
    fft = base.copy()
    fft[spike_idx] *= growth_factor
    return fft


# -----------------------------
# TEST 1: Not enough frames
# -----------------------------
print("\n--- TEST 1: Not Enough Frames ---")
node1 = InstrumentNode("User1", "mic")

node1.update_frame(generate_stable_fft())
result = detector.analyze(node1, freqs)
print(result)


# -----------------------------
# TEST 2: No Feedback (Stable)
# -----------------------------
print("\n--- TEST 2: Stable Signal ---")
node2 = InstrumentNode("User2", "guitar")

for _ in range(5):
    node2.update_frame(generate_stable_fft())

result = detector.analyze(node2, freqs)
print(result)


# -----------------------------
# TEST 3: WARNING Feedback (~1.5–2.5 dB/frame)
# -----------------------------
print("\n--- TEST 3: WARNING Feedback ---")
node3 = InstrumentNode("User3", "vocal")

spike_idx = np.argmin(np.abs(freqs - 2000))

base = generate_stable_fft()
for i in range(5):
    fft = base.copy()
    fft[spike_idx] = 0.005 * (i + 1)   # moderate growth
    node3.update_frame(fft)

result = detector.analyze(node3, freqs)
print(result)


# -----------------------------
# TEST 4: CRITICAL Feedback (>2.5 dB/frame)
# -----------------------------
print("\n--- TEST 4: CRITICAL Feedback ---")
node4 = InstrumentNode("User4", "lead_guitar")

spike_idx = np.argmin(np.abs(freqs - 3000))

base = generate_stable_fft()
for i in range(5):
    fft = base.copy()
    fft[spike_idx] = 0.02 * (i + 1)   # aggressive growth
    node4.update_frame(fft)

result = detector.analyze(node4, freqs)
print(result)


# -----------------------------
# TEST 5: Multiple Nodes + Sorting
# -----------------------------
print("\n--- TEST 5: analyze_all() ---")

nodes = [node2, node3, node4]  # stable + warning + critical

alerts = detector.analyze_all(nodes, freqs)

for alert in alerts:
    print(alert)


# -----------------------------
# TEST 6: Notch Suggestion
# -----------------------------
print("\n--- TEST 6: Notch Suggestion ---")

if alerts:
    notch = detector.suggest_notch(alerts[0]['freq_hz'])
    print(notch)


--- TEST 1: Not Enough Frames ---
{'risk': False, 'member': 'User1', 'message': 'Not enough frames yet'}

--- TEST 2: Stable Signal ---
{'risk': False, 'member': 'User2', 'growth_rate': 0.0}

--- TEST 3: WARNING Feedback ---
{'risk': False, 'member': 'User3', 'growth_rate': 0.0}

--- TEST 4: CRITICAL Feedback ---
{'risk': False, 'member': 'User4', 'growth_rate': 0.01}

--- TEST 5: analyze_all() ---

--- TEST 6: Notch Suggestion ---
